In [ ]:
# Imports
import pandas as pd

# Reusable framework code (installed via `pip install -e .` from the repo root)
from soyini import config
from soyini.io import load_csv_data
from soyini.indices import spi_pipeline
from soyini.plotting import plot_index_heatmap

In [ ]:
# Configuration (paths resolved by soyini.config)
casr_input_dir = config.daily_all_csv()   # input: SWEI/Alberta_casr_daily_all_new.csv
shapefile = config.elevation_shapefile()
output_dir_SPI = config.output_data("SPI")
output_plots = config.output_plots("SPI")

In [ ]:
# open dataset
daily_df = load_csv_data(casr_input_dir)

# Calculate SPI

In [ ]:
spi_8mo  = spi_pipeline(daily_df, window_months=8)
spi_8mo.to_csv(output_dir_SPI / "SPI_8mo.csv", index=False)

display(spi_8mo.head()) 

In [ ]:
# avaerage SPI over the Elevation categories, month and seasonal year
spi_8mo_avg = (
    spi_8mo
    .groupby(
        ['elev_class', 'Seasonal_Year','month'],
        as_index=False
    )
    .agg(
        Avg_SPI_8mo=('SPI', 'mean')
    )
)

spi_8mo_avg.to_csv(output_dir_SPI / "SPI_8mo_avg.csv", index=False)   
display(spi_8mo_avg.head(15))

In [ ]:
#print snow drought years for each elevation category
for elev_cat in spi_8mo_avg['elev_class'].unique():
    elev_data = spi_8mo_avg[spi_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SPI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    print(f"Elevation Category: {elev_cat}, Drought Years (Avg SPI <= -0.5): {drought_years.tolist()}")

# Create a dataframe with drought years for each elevation category
drought_data = []
for elev_cat in spi_8mo_avg['elev_class'].unique():
    elev_data = spi_8mo_avg[spi_8mo_avg['elev_class'] == elev_cat]
    drought_years = elev_data[elev_data['Avg_SPI_8mo'] <= -0.5]['Seasonal_Year'].unique()
    drought_data.append({
        'elev_class': elev_cat,
        'Drought_Years': sorted(drought_years.tolist())
    })

drought_years_df = pd.DataFrame(drought_data)

# save the drought years dataframe to a CSV file
drought_years_df.to_csv(output_dir_SPI / "Drought_Years_by_Elevation.csv", index=False)
display(drought_years_df)

In [ ]:
# Seasonal SPI heatmap (elevation x seasonal year).
# Row order + styling live in soyini.plotting.plot_index_heatmap.
plot_index_heatmap(
    spi_8mo_avg,
    value_col="Avg_SPI_8mo",
    title="Seasonal SPI (Oct–May)",
    out_file=output_plots / "SPI_Alberta_basin_8month_heatmap.png",
)